# Part 1: E-Commerce Customer Segmentation using RFM and Clustering
**Business Analyst Project | K-Means Clustering on RFM Features**

---
**Business Problem:** An e-commerce company wants to segment its customers to design targeted marketing campaigns, improve retention, and identify high-value customer groups. Using transaction-level data, we will build customer-level RFM features and apply K-Means clustering to identify distinct customer segments.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Libraries imported successfully.')

## 2. Load Dataset

In [ ]:
df_raw = pd.read_csv('part_1_ecommerce_customer_segmentation.csv')
print('Shape:', df_raw.shape)
df_raw.head()

## 3. Data Understanding

In [ ]:
print('=== Column Info ===')
print(df_raw.dtypes)
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Basic Statistics ===')
df_raw.describe()

### Data Understanding Summary

| Column | Description |
|---|---|
| InvoiceNo | Unique identifier for each transaction/invoice |
| StockCode | Unique product code |
| Description | Name/description of the product |
| Category | Product category (e.g. Electronics, Grocery) |
| Quantity | Number of units purchased in the transaction |
| InvoiceDate | Date and time of the transaction |
| UnitPrice | Price per unit of the product |
| CustomerID | Unique identifier for each customer |
| Country | Country where the customer is located |

**Each row** represents a single line item within an invoice — one product purchased in one transaction by one customer.

**Business Type:** Multi-category e-commerce retailer operating across 11 countries.

**What can be answered:**
- Which customers are most valuable?
- Which products/categories drive the most revenue?
- Which countries are the top markets?
- How frequently do customers buy?

**What cannot be answered (missing data):**
- Customer demographics (age, gender)
- Marketing channel or source of acquisition
- Product return reasons

## 4. Data Cleaning

In [ ]:
df = df_raw.copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Step 1: Drop missing CustomerID (cannot attribute purchase to any customer)
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f'Step 1 — Dropped {before - len(df)} rows with missing CustomerID')

# Step 2: Drop missing Description
before = len(df)
df = df.dropna(subset=['Description'])
print(f'Step 2 — Dropped {before - len(df)} rows with missing Description')

# Step 3: Remove negative/zero quantities (returns or data errors)
before = len(df)
df = df[df['Quantity'] > 0]
print(f'Step 3 — Removed {before - len(df)} rows with non-positive Quantity')

# Step 4: Remove zero/negative unit prices
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f'Step 4 — Removed {before - len(df)} rows with non-positive UnitPrice')

# Step 5: Remove cancelled invoices (prefix 'C')
before = len(df)
df = df[~df['InvoiceNo'].str.startswith('C')]
print(f'Step 5 — Removed {before - len(df)} cancelled invoices')

# Step 6: Remove duplicate records
before = len(df)
df = df.drop_duplicates()
print(f'Step 6 — Removed {before - len(df)} duplicate rows')

# Step 7: Add Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(f'\nFinal clean dataset: {df.shape[0]} rows, {df["CustomerID"].nunique()} unique customers')

## 5. Feature Engineering — RFM + Customer-Level Features

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date (reference for Recency):', snapshot_date)

rfm = df.groupby('CustomerID').agg(
    Recency        = ('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency      = ('InvoiceNo',    'nunique'),
    Monetary       = ('Revenue',      'sum'),
    TotalQty       = ('Quantity',     'sum'),
    AvgOrderValue  = ('Revenue',      'mean'),
    UniqueProducts = ('StockCode',    'nunique'),
    Country        = ('Country',      'first')
).reset_index()

print('RFM Table Shape:', rfm.shape)
rfm.head(10)

## 6. Exploratory Data Analysis

In [ ]:
# Chart 1: Top 10 Countries by Revenue
fig, ax = plt.subplots(figsize=(10,5))
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=ax, palette='Blues_r')
ax.set_title('Top 10 Countries by Total Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.set_ylabel('Country')
for i, v in enumerate(top_countries.values):
    ax.text(v + 100, i, f'£{v:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/01_top_countries_revenue.png')
plt.show()
print('Interpretation: The top markets dominate revenue. Country-specific strategies should be designed for the leading nations.')

In [ ]:
# Chart 2: Top 10 Products by Quantity Sold
fig, ax = plt.subplots(figsize=(10,5))
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_products.values, y=top_products.index, ax=ax, palette='Greens_r')
ax.set_title('Top 10 Products by Quantity Sold', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Quantity Sold')
ax.set_ylabel('Product')
plt.tight_layout()
plt.savefig('images/02_top_products_quantity.png')
plt.show()
print('Interpretation: A small number of products account for most volume. These items should always be kept well-stocked.')

In [ ]:
# Chart 3: Top 10 Products by Revenue
fig, ax = plt.subplots(figsize=(10,5))
top_rev_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_rev_products.values, y=top_rev_products.index, ax=ax, palette='Oranges_r')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.set_ylabel('Product')
plt.tight_layout()
plt.savefig('images/03_top_products_revenue.png')
plt.show()
print('Interpretation: High-revenue products may differ from high-quantity products, indicating premium items that deserve special promotion.')

In [ ]:
# Chart 4: Distribution of Purchase Frequency
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(rfm['Frequency'], bins=30, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Customer Purchase Frequency', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('images/04_frequency_distribution.png')
plt.show()
print('Interpretation: Most customers have a low purchase frequency, suggesting the majority are occasional buyers rather than loyals.')

In [ ]:
# Chart 5: Distribution of Monetary Value
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(rfm['Monetary'], bins=40, color='coral', edgecolor='white')
ax.set_title('Distribution of Customer Monetary Value (Total Spend)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Spend (£)')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('images/05_monetary_distribution.png')
plt.show()
print('Interpretation: Spend is right-skewed — a small group of customers accounts for a disproportionately large share of revenue.')

In [ ]:
# Chart 6: Boxplots — Outlier Detection
fig, axes = plt.subplots(1,3, figsize=(13,5))
for ax, col, color in zip(axes, ['Quantity','UnitPrice','Revenue'], ['#4C72B0','#DD8452','#55A868']):
    ax.boxplot(df[col], patch_artist=True, boxprops=dict(facecolor=color, alpha=0.6))
    ax.set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    ax.set_ylabel(col)
fig.suptitle('Outlier Detection in Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/06_boxplots_outliers.png')
plt.show()
print('Interpretation: Significant outliers exist across quantity, price, and revenue. These likely represent bulk or wholesale orders.')

In [ ]:
# Chart 7: Revenue by Category
fig, ax = plt.subplots(figsize=(10,5))
cat_rev = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
sns.barplot(x=cat_rev.values, y=cat_rev.index, ax=ax, palette='Set2')
ax.set_title('Total Revenue by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig('images/07_revenue_by_category.png')
plt.show()
print('Interpretation: Certain categories dominate revenue. The company should invest more in promoting top-performing categories.')

## 7. Model Building — K-Means Clustering on RFM

In [ ]:
# Select features and normalize
features = rfm[['Recency','Frequency','Monetary']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
print('Features scaled. Shape:', X_scaled.shape)

# Elbow Method + Silhouette Score
inertias, sil_scores = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_title('Elbow Method — Optimal K', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia')
ax1.axvline(x=4, color='red', linestyle='--', label='K=4 (chosen)')
ax1.legend()
ax2.plot(K_range, sil_scores, 'rs-', linewidth=2, markersize=8)
ax2.set_title('Silhouette Score by K', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.axvline(x=4, color='red', linestyle='--', label='K=4 (chosen)')
ax2.legend()
plt.tight_layout()
plt.savefig('images/08_elbow_silhouette.png')
plt.show()
print('Interpretation: The elbow method shows a clear bend at K=4. Silhouette scores confirm K=4 as a strong choice for meaningful separation.')

In [ ]:
# Train final K-Means model with K=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(X_scaled)
print('Cluster labels assigned.')
print('Cluster sizes:')
print(rfm['Cluster'].value_counts().sort_index())

## 8. Evaluation — Cluster Profiles

In [ ]:
# RFM Profile per Cluster
cluster_profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(1)
print('=== Average RFM per Cluster ===')
print(cluster_profile)

# Chart 9: Cluster Scatter Plot
colors = ['#E74C3C','#2ECC71','#3498DB','#F39C12']
fig, ax = plt.subplots(figsize=(9,6))
for i in range(4):
    mask = rfm['Cluster'] == i
    ax.scatter(rfm.loc[mask,'Recency'], rfm.loc[mask,'Monetary'],
               label=f'Cluster {i}', alpha=0.7, s=60, color=colors[i])
ax.set_title('Customer Clusters: Recency vs Monetary Value', fontsize=14, fontweight='bold')
ax.set_xlabel('Recency (days since last purchase)')
ax.set_ylabel('Monetary Value (Total Spend £)')
ax.legend(title='Cluster')
plt.tight_layout()
plt.savefig('images/09_clusters_scatter.png')
plt.show()
print('Interpretation: Clusters are clearly separated by spending level and recency. High-spend customers cluster distinctly from occasional or inactive ones.')

In [ ]:
# Chart 10: RFM Bar Charts per Cluster
fig, axes = plt.subplots(1, 3, figsize=(14,5))
pal = ['#E74C3C','#2ECC71','#3498DB','#F39C12']
for ax, metric in zip(axes, ['Recency','Frequency','Monetary']):
    vals = cluster_profile[metric]
    bars = ax.bar(vals.index, vals.values, color=pal)
    ax.set_title(f'Avg {metric} by Cluster', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.0f}', ha='center', va='bottom', fontsize=10)
plt.suptitle('RFM Profile per Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/10_cluster_rfm_profiles.png')
plt.show()
print('Interpretation: Cluster 3 has the highest frequency and monetary value. Cluster 2 has the highest recency (inactive). Clusters 0 and 1 are mid-range.')

In [ ]:
# Chart 11: Cluster Sizes
fig, ax = plt.subplots(figsize=(7,5))
sizes = rfm['Cluster'].value_counts().sort_index()
ax.bar(sizes.index, sizes.values, color=pal)
ax.set_title('Number of Customers per Cluster', fontsize=14, fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_ylabel('Customer Count')
for i, v in enumerate(sizes.values):
    ax.text(i, v + 2, str(v), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('images/11_cluster_sizes.png')
plt.show()
print('Interpretation: Cluster 0 is the largest segment (occasional buyers). Cluster 3 is the smallest but most valuable group.')

## 9. Business Insights — Cluster Interpretation

| Cluster | Recency | Frequency | Monetary | Customer Type | Business Value |
|---|---|---|---|---|---|
| 0 | ~74 days | ~1.5 orders | ~£8,192 | **Occasional Buyers** | Large group; low engagement but moderate spend per order |
| 1 | ~71 days | ~3.2 orders | ~£19,588 | **Regular Customers** | Recently active, buy often — valuable growth segment |
| 2 | ~233 days | ~1.4 orders | ~£8,242 | **Inactive / At-Risk** | Haven't bought in 7+ months — churn risk |
| 3 | ~77 days | ~3.9 orders | ~£46,557 | **High-Value Loyals** | Top spenders, frequent buyers — company's most valuable customers |


## 10. Final Recommendations

### Cluster 3 — High-Value Loyal Customers (44 customers, avg spend £46,557)
- Launch a **VIP loyalty programme** with exclusive early access to new products, free express shipping, and dedicated account managers.
- Send personalised product bundles based on their top purchased categories.
- Prevent churn with a **proactive retention alert**: if a Cluster 3 customer goes 30+ days without purchasing, trigger a personal outreach.

### Cluster 1 — Regular Customers (163 customers, avg spend £19,588)
- These customers are already engaged — the goal is to **upgrade them to Cluster 3**.
- Offer a **"spend and earn" rewards scheme**: spend £X more this month to unlock premium tier benefits.
- Use cross-sell recommendations based on their most purchased categories.

### Cluster 0 — Occasional Buyers (257 customers, avg spend £8,192)
- Run **seasonal or event-based promotions** (e.g. holiday sales, flash deals) to bring them back.
- Send a "We miss you" email after 45 days of inactivity with a limited-time discount voucher.
- Highlight popular products in their preferred category to reduce friction to purchase.

### Cluster 2 — Inactive / At-Risk Customers (217 customers, avg spend £8,242)
- These customers have not purchased in ~233 days on average — immediate **re-engagement campaigns** are needed.
- Send a targeted win-back email with a **15–20% discount code** valid for 2 weeks.
- If no response after 2 re-engagement attempts, consider moving them to a low-frequency newsletter list to reduce unsubscribes.
- Investigate if these customers share a geographic pattern (e.g. specific countries) to determine if regional factors caused churn.


In [ ]:
# Save final RFM + cluster table
rfm.to_csv('outputs/rfm_clustered_customers.csv', index=False)
print('Final RFM table with cluster labels saved to outputs/rfm_clustered_customers.csv')
print('\n=== Project Complete ===')